# Colab quickstart for the TRPO / PPO repo

This notebook is a **reference / helper notebook** for running the repository on Google Colab.

It is designed around the workflow we settled on during development:

1. keep the canonical repo on Google Drive
2. copy it into `/content/trpo` at the start of a Colab session
3. run experiments from local SSD (`/content`)
4. optionally save outputs back to Drive afterward

The examples below are intentionally simple and meant to be copied / adapted.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Copy the repo from Drive to Colab local SSD

Edit the Drive path below to match your own folder structure.

In [ ]:
!rm -rf /content/trpo
!cp -r "/content/drive/MyDrive/Colab Notebooks/839/trpo" /content/trpo
%cd /content/trpo

## 3. Install dependencies inside the Colab runtime

The repository scripts bootstrap the local `/content/trpo` checkout, so the dependency install is enough when running commands from the repo root.

In [ ]:
!python3 -m pip install -r requirements.txt

## 4. Check whether CUDA is available

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")

## 5. Example: locomotion run

This is a simple example for a locomotion task.

In [ ]:
!python3 scripts/train.py   --config configs/mujoco/swimmer_single_path.yaml   --device cuda   --progress-mode notebook   --overwrite

## 6. Example: Atari TRPO run on Colab

This uses the practical Colab setup that worked well during development:

- `memory_mode safe`
- `obs_storage ram`
- tuned `full_batch_chunk_size`
- optional Fisher-vector-product subsampling
- CUDA
- local output directory under `/content/...`

In [ ]:
!python3 scripts/train.py   --config configs/atari/seaquest_single_path.yaml   --memory-mode safe   --obs_storage ram   --full_batch_chunk_size 8192   --fvp_subsample_fraction 0.1   --device cuda   --progress-mode off   --output-dir /content/trpo_runs/seaquest_single_path/seed_0   --overwrite

## 7. Example: Atari PPO run on Colab

In [ ]:
!python3 scripts/train.py   --config configs/atari/seaquest_ppo_clip.yaml   --memory-mode standard   --device cuda   --progress-mode off   --output-dir /content/trpo_runs/seaquest_ppo_clip/seed_0   --overwrite

## 8. Paper-style batch launcher

This preserves the lightweight paper-batch launcher that used to live in a standalone script.

In [ ]:
import subprocess
import sys

PAPER_SUITES = {
    "mujoco": [
        "configs/mujoco/swimmer_single_path.yaml",
        "configs/mujoco/hopper_single_path.yaml",
        "configs/mujoco/walker2d_single_path.yaml",
    ],
    "atari": [
        "configs/atari/beamrider_single_path.yaml",
        "configs/atari/breakout_single_path.yaml",
        "configs/atari/enduro_single_path.yaml",
        "configs/atari/pong_single_path.yaml",
        "configs/atari/qbert_single_path.yaml",
        "configs/atari/seaquest_single_path.yaml",
        "configs/atari/spaceinvaders_single_path.yaml",
    ],
}

def run_paper_suite(suite, seeds=(0,), device="cuda", extra_args=()):
    for config in PAPER_SUITES[suite]:
        for seed in seeds:
            cmd = [
                sys.executable,
                "scripts/train.py",
                "--config",
                config,
                "--seed",
                str(seed),
                "--device",
                device,
                *extra_args,
            ]
            print("Running:", " ".join(cmd))
            subprocess.run(cmd, check=True)

# Uncomment one when you are ready.
# run_paper_suite("mujoco", seeds=(0,), device="cuda")
# run_paper_suite("atari", seeds=(0,), device="cuda", extra_args=("--memory-mode", "safe", "--obs-storage", "ram", "--progress-mode", "off"))

## 9. Aggregate runs

In [ ]:
!python3 scripts/aggregate_results.py   --runs-root outputs/swimmer_single_path   --runs-root outputs/swimmer_natural_pg   --runs-root outputs/swimmer_ppo_clip   --compare   --metric train_return_mean   --x-axis iteration

## 10. Copy results back to Drive

If you wrote outputs to `/content/...`, copy them back to Drive when the run finishes.

In [ ]:
!mkdir -p "/content/drive/MyDrive/Colab Notebooks/839/trpo_outputs"
!cp -r /content/trpo_runs "/content/drive/MyDrive/Colab Notebooks/839/trpo_outputs/"

## 11. Notes

- For **Atari**, start with `memory_mode safe`.
- If Colab has plenty of RAM, `obs_storage ram` is usually faster than memmap.
- Tune `full_batch_chunk_size` conservatively: `4096`, `8192`, then maybe `16384`.
- If using TRPO/NPG, `fvp_subsample_fraction 0.1` is a useful optional speedup.
- Run from `/content/trpo`, not directly from mounted Drive, for better performance.